# Final Evaluation, Hypothesis Decisions, and Project Synthesis

**SoftUni Deep Learning Final Project**  
**Notebook:** `09_final_evaluation.ipynb`

This notebook consumes the canonical result packages exported by Notebooks 04–08. It does not retrain models. Its role is to validate artifact completeness, consolidate comparable results, evaluate the six predefined hypotheses, and export the evidence required for the final written report.

Static normalized-price models and contract-level Longstaff–Schwartz methods are reported separately because they operate on different evaluation units and price scales.


## 1. Environment and strict final-evaluation configuration

`STRICT_MODE = True` is the submission setting. Missing or schema-incompatible required artifacts stop execution instead of producing polished but empty placeholder files.


In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == "notebooks"
    else NOTEBOOK_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.artifact_registry import audit_artifacts
from src.evaluation.final_project_evaluation import (
    build_boundary_comparison,
    build_financial_consistency_table,
    build_hypothesis_evidence,
    build_literature_handoff,
    build_lsm_comparison,
    build_metric_inventory,
    build_ood_comparison,
    build_runtime_comparison_from_results,
    build_static_ablation_table,
    build_static_pricing_table,
    load_project_results,
)
from src.evaluation.final_reporting import (
    export_final_evaluation,
    write_json,
)
from src.evaluation.hypothesis_testing import (
    decide_all_hypotheses,
)

STRICT_MODE = True
FINAL_OUTPUT_DIR = (
    PROJECT_ROOT / "artifacts" / "final_evaluation"
)
FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DESIGN_PATH = (
    PROJECT_ROOT
    / "data"
    / "manifests"
    / "final_evaluation_design.json"
)

design = (
    json.loads(DESIGN_PATH.read_text(encoding="utf-8"))
    if DESIGN_PATH.is_file()
    else {
        "expected_total_observations": 1_450_000,
        "artifact_version": "canonical_final_evaluation_v2",
    }
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Strict mode:  {STRICT_MODE}")
print(f"Output dir:   {FINAL_OUTPUT_DIR}")
display(pd.Series(design, name="value").to_frame())


Project root: /storage/deep_learning_american_option_pricing
Strict mode:  True
Output dir:   /storage/deep_learning_american_option_pricing/artifacts/final_evaluation


,value
artifact_version,step9_final_evaluation_skeleton_v1
expected_total_observations,1450000
notebook,notebooks/09_final_evaluation.ipynb
markdown_twin,docs/09_final_evaluation.md
output_directory,artifacts/final_evaluation
model_registry,"[black_scholes_proxy, crr_benchmark, direct_ml..."
hypothesis_rules,{'H1': 'direct_mlp_mae / black_scholes_mae <= ...
required_outputs,"[consolidated_model_metrics.csv, metric_invent..."


## 2. Canonical artifact audit

The audit validates exact files, JSON keys, table columns, row presence, completion status, and canonical selected checkpoints. Notebook 09's own previous exports are deliberately excluded from the upstream registry.


In [2]:
artifact_audit = audit_artifacts(PROJECT_ROOT)

display(
    artifact_audit[
        [
            "name",
            "category",
            "required_for_final",
            "found",
            "valid",
            "resolved_path",
            "notes",
        ]
    ]
)

required_audit = artifact_audit.loc[
    artifact_audit["required_for_final"]
].copy()

invalid_required = required_audit.loc[
    ~required_audit["valid"]
].copy()

valid_required = int(required_audit["valid"].sum())
total_required = int(len(required_audit))

print(
    f"Valid required artifacts: "
    f"{valid_required}/{total_required}"
)

if STRICT_MODE and not invalid_required.empty:
    missing_summary = "\n".join(
        f"- {row.name}: {row.notes}"
        for row in invalid_required[
            ["name", "notes"]
        ].itertuples(index=False)
    )
    raise RuntimeError(
        "Final evaluation cannot continue because required "
        "artifacts are missing or invalid:\n"
        + missing_summary
        + "\nRun the canonical export cells in Notebooks "
        "04–08 or execute `dvc pull`."
    )


,name,category,required_for_final,found,valid,resolved_path,notes
0,production_dataset_manifest,data,True,True,True,/storage/deep_learning_american_option_pricing...,JSON schema validated
1,direct_final_metrics,notebook_04,True,True,True,/storage/deep_learning_american_option_pricing...,JSON schema validated
2,direct_test_predictions,notebook_04,True,True,True,/storage/deep_learning_american_option_pricing...,table schema validated
3,premium_final_metrics,notebook_05,True,True,True,/storage/deep_learning_american_option_pricing...,JSON schema validated
4,premium_test_predictions,notebook_05,True,True,True,/storage/deep_learning_american_option_pricing...,table schema validated
5,premium_checkpoint,checkpoint,True,True,True,/storage/deep_learning_american_option_pricing...,artifact exists and is non-empty
6,multitask_final_metrics,notebook_06,True,True,True,/storage/deep_learning_american_option_pricing...,JSON schema validated
7,multitask_test_predictions,notebook_06,True,True,True,/storage/deep_learning_american_option_pricing...,table schema validated
8,multitask_checkpoint,checkpoint,True,True,True,/storage/deep_learning_american_option_pricing...,artifact exists and is non-empty
9,lsm_final_metrics,notebook_07,True,True,True,/storage/deep_learning_american_option_pricing...,JSON schema validated


Valid required artifacts: 22/22


## 3. Load canonical packages and build the metric inventory

Every model family has an explicit adapter. Metrics are no longer inferred by searching for any key that happens to end in `mae` or `rmse`.


In [3]:
project_results = load_project_results(PROJECT_ROOT)
metric_inventory = build_metric_inventory(PROJECT_ROOT)

package_status = pd.DataFrame(
    [
        {
            "package": name,
            "loaded": payload is not None,
            "payload_type": type(payload).__name__,
        }
        for name, payload in project_results.items()
    ]
)

display(package_status)
display(metric_inventory.head(40))
print(
    f"Metric inventory rows: {len(metric_inventory):,}"
)


,package,loaded,payload_type
0,direct,True,dict
1,premium,True,dict
2,multitask,True,dict
3,lsm,True,dict
4,integrated,True,dict


,source,metric,value
0,production_dataset_manifest,manifest_version,1
1,production_dataset_manifest,created_at_utc,2026-07-27T12:06:25.134516+00:00
2,production_dataset_manifest,project,deep_learning_american_option_pricing
3,production_dataset_manifest,generation_config.core_observations,1000000
4,production_dataset_manifest,generation_config.boundary_observations,250000
5,production_dataset_manifest,generation_config.ood_observations_per_set,50000
6,production_dataset_manifest,generation_config.tree_steps,250
7,production_dataset_manifest,generation_config.strike,100.0
8,production_dataset_manifest,generation_config.chunk_size,25000
9,production_dataset_manifest,generation_config.seed,42


Metric inventory rows: 1,871


## 4. Consolidated in-domain static pricing comparison

This table contains only models evaluated on the common static normalized-price test split. Lower MAE and RMSE are better. Longstaff–Schwartz results are excluded here because they use contract-level raw prices and Monte Carlo confidence intervals.


In [4]:
static_pricing = build_static_pricing_table(
    project_results
)

if STRICT_MODE and static_pricing.empty:
    raise RuntimeError(
        "Static pricing table is empty."
    )

display(static_pricing)

best_static = static_pricing.loc[
    static_pricing["mae"].idxmin()
]
print(
    "Lowest static test MAE: "
    f"{best_static['model']} "
    f"({best_static['mae']:.8f})"
)


,model,source_notebook,observations,mae,rmse,median_absolute_error,max_absolute_error,mean_error,r2,within_0.001,within_0.005,within_0.01,within_0.05,below_european_count,below_european_rate,below_intrinsic_count,below_intrinsic_rate,negative_price_count,negative_price_rate,total_bound_violations
0,Constrained floor residual,05,187811.0,0.000102,0.000213,0.000046,0.004641,-1.426903e-05,0.999998,0.992860,1.000000,1.000000,1.000000,0.0,0.000000,0.0,0.000000,0.0,0.00000,0.0
1,Price-only constrained residual,06,187811.0,0.000102,0.000213,0.000046,0.004641,-1.426903e-05,0.999998,0.992860,1.000000,1.000000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Non-negative premium,05,187811.0,0.000205,0.000356,0.000115,0.006268,2.815696e-05,0.999995,0.975859,0.999925,1.000000,1.000000,0.0,0.000000,23710.0,0.126244,0.0,0.00000,23710.0
3,Unconstrained premium,05,187811.0,0.000220,0.000359,0.000142,0.007582,-1.652149e-05,0.999995,0.977951,0.999867,1.000000,1.000000,25968.0,0.138267,31525.0,0.167855,2727.0,0.01452,60220.0
4,Multi-task constrained residual,06,187811.0,0.000242,0.000504,0.000082,0.010295,1.801177e-07,0.999990,0.952404,0.999287,0.999989,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Final integrated constrained price,08,187811.0,0.000562,0.001122,0.000132,0.013950,3.683461e-04,0.999952,0.811065,0.992327,0.999878,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Direct MLP,05,187811.0,0.000782,0.001146,0.000548,0.019201,-1.283024e-04,0.999950,0.741927,0.995107,0.999766,1.000000,25817.0,0.137463,32993.0,0.175671,0.0,0.00000,58810.0
7,Final integrated direct head,08,187811.0,0.005621,0.007370,0.004524,0.046731,-8.286418e-04,0.997928,0.126995,0.542998,0.852932,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Black–Scholes proxy,04,187811.0,0.013398,0.027384,0.002710,0.188955,-1.339839e-02,0.971395,0.390595,0.583757,0.690039,0.917438,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Zero premium,05,187811.0,0.013398,0.027384,0.002710,0.188955,-1.339839e-02,0.971395,0.390595,0.583757,0.690039,0.917438,0.0,0.000000,57780.0,0.307650,0.0,0.00000,57780.0


Lowest static test MAE: Constrained floor residual (0.00010187)


## 5. Financial-consistency comparison

The table combines empirical lower-bound violations and integrated-model internal-consistency diagnostics. Architectural guarantees are distinguished from empirical monotonicity checks.


In [5]:
financial_consistency = (
    build_financial_consistency_table(
        project_results
    )
)

if STRICT_MODE and financial_consistency.empty:
    raise RuntimeError(
        "Financial-consistency table is empty."
    )

display(financial_consistency)


,model,source_notebook,negative_price_violations,negative_price_violation_rate,below_intrinsic_violations,below_intrinsic_violation_rate,below_european_violations,below_european_violation_rate,below_european_count,below_european_rate,...,exercise_probability_rmse,decision_disagreement_rate,constrained_negative_rate,constrained_below_european_rate,constrained_below_intrinsic_rate,direct_negative_rate,direct_below_european_rate,direct_below_intrinsic_rate,any_contradiction_rate,residual_reconstruction_mae
0,Constrained floor residual,05,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Non-negative premium,05,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Unconstrained premium,05,NaN,NaN,NaN,NaN,NaN,NaN,25968.0,0.138267,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Direct MLP,05,NaN,NaN,NaN,NaN,NaN,NaN,25817.0,0.137463,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Zero premium,05,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Mean premium,05,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Price-only constrained residual,06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Multi-task constrained residual,06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Final integrated multi-head,08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.041942,0.006799,0.0,0.0,0.0,0.0,0.163579,0.267689,0.385718,3.529938e-09


## 6. Exercise-boundary and classification comparison

Notebook 06 provides the standalone and multi-task boundary comparison. Notebook 08 adds the final integrated exercise head and the corrected boundary-band analysis. F1 and balanced accuracy are left undefined for bands containing only one actual class.


In [6]:
boundary_comparison = build_boundary_comparison(
    project_results
)

if STRICT_MODE and boundary_comparison.empty:
    raise RuntimeError(
        "Boundary comparison is empty."
    )

display(boundary_comparison)


,model,source_notebook,accuracy,balanced_accuracy,brier_score,f1,observations,positive_rate,pr_auc,precision,recall,roc_auc,threshold,boundary_location_error,near_boundary_observations,near_boundary_price_mae,near_boundary_exercise_accuracy,near_boundary_exercise_f1,near_boundary_balanced_accuracy
0,Exercise-only classifier,06,0.998264,0.997874,0.003694,0.996393,187811.0,0.240454,0.999946,0.995666,0.997121,0.999982,0.670,0.001117,NaN,NaN,NaN,NaN,NaN
1,Multi-task model,06,0.998264,0.997344,0.003922,0.996388,187811.0,0.240454,0.999934,0.997205,0.995571,0.999979,0.685,0.000972,NaN,NaN,NaN,NaN,NaN
2,Final integrated exercise head,08,0.993557,0.995532,0.004480,0.986772,187811.0,0.240454,0.999901,0.974520,0.999336,0.999968,0.500,NaN,59673.0,0.000034,0.97974,0.986782,0.959049


## 7. Out-of-domain deterioration

OOD MAE is compared with each model's own in-domain MAE. The aggregate hypothesis decision is based on one selected static model, while the full table preserves regime-specific deterioration.


In [7]:
ood_results = build_ood_comparison(
    project_results,
    static_pricing,
)

if STRICT_MODE and ood_results.empty:
    raise RuntimeError(
        "OOD comparison is empty."
    )

display(ood_results)

ood_summary = (
    ood_results.groupby("model", as_index=False)
    .agg(
        regimes=("regime", "nunique"),
        mean_ood_mae=("ood_mae", "mean"),
        mean_deterioration=(
            "ood_deterioration",
            "mean",
        ),
    )
    .sort_values("mean_ood_mae")
)

display(ood_summary)


,model,regime,observations,in_domain_mae,ood_mae,ood_rmse,ood_deterioration,source_notebook
0,Constrained floor residual,extreme_moneyness,50000.0,0.000102,0.000118,0.000378,0.158667,05
1,Constrained floor residual,high_volatility,50000.0,0.000102,0.000568,0.001110,4.575369,05
2,Constrained floor residual,long_maturity,50000.0,0.000102,0.003432,0.007965,32.693782,05
3,Constrained floor residual,rate_dividend,50000.0,0.000102,0.002961,0.006051,28.067321,05
4,Direct MLP,extreme_moneyness,50000.0,0.000782,0.011993,0.020319,14.342276,04
5,Direct MLP,extreme_moneyness,50000.0,0.000782,0.011993,0.020319,14.342276,05
6,Direct MLP,high_volatility,50000.0,0.000782,0.009221,0.014600,10.796699,04
7,Direct MLP,high_volatility,50000.0,0.000782,0.009221,0.014600,10.796699,05
8,Direct MLP,long_maturity,50000.0,0.000782,0.046026,0.087061,57.881747,04
9,Direct MLP,long_maturity,50000.0,0.000782,0.046026,0.087061,57.881747,05


,model,regimes,mean_ood_mae,mean_deterioration
0,Constrained floor residual,4,0.001770,16.373785
2,Final integrated constrained price,4,0.003204,4.705425
4,Non-negative premium,4,0.003641,16.755294
3,Multi-task constrained residual,4,0.004840,18.995343
1,Direct MLP,4,0.019690,24.189107


## 8. Runtime and computational efficiency

Marginal static inference, per-contract numerical valuation, and one-time training cost are separate cost categories. They are not blended into one misleading speed ranking.


In [8]:
runtime_comparison = (
    build_runtime_comparison_from_results(
        project_results
    )
)

if STRICT_MODE and runtime_comparison.empty:
    raise RuntimeError(
        "Runtime comparison is empty."
    )

display(runtime_comparison)


,model,device,observations,seconds,source_notebook,cost_type,benchmark_contracts,seconds_per_observation,observations_per_second
0,Constrained floor residual,cuda,100000,0.001203,05,marginal inference,NaN,1.202729e-08,8.314425e+07
1,Direct MLP,cuda,100000,0.001213,04,marginal inference,NaN,1.212697e-08,8.246083e+07
2,Direct MLP,cuda,100000,0.001297,05,marginal inference,NaN,1.296705e-08,7.711854e+07
3,Non-negative premium,cuda,100000,0.001320,05,marginal inference,NaN,1.320169e-08,7.574788e+07
4,Final integrated multi-head,cuda,100000,0.002953,08,marginal inference,NaN,2.952898e-08,3.386504e+07
5,Direct MLP,cuda,10000,0.000439,04,marginal inference,NaN,4.393450e-08,2.276115e+07
6,Direct MLP,cuda,1000,0.000467,04,marginal inference,NaN,4.666670e-07,2.142856e+06
7,Neural LSM evaluation,None,1,0.075390,07,per-contract valuation,100.0,7.538985e-02,1.326439e+01
8,Classical LSM,None,1,0.082729,07,per-contract valuation,100.0,8.272901e-02,1.208766e+01
9,CRR,None,1,0.090855,07,per-contract valuation,100.0,9.085546e-02,1.100649e+01


## 9. Static-model ablation

The ablation table links model structure with observed test MAE, exercise quality, and available financial-consistency evidence.


In [9]:
static_ablation = build_static_ablation_table(
    static_pricing,
    financial_consistency,
    boundary_comparison,
)

if STRICT_MODE and static_ablation.empty:
    raise RuntimeError(
        "Static ablation table is empty."
    )

display(static_ablation)


,model,direct_learning,residual_learning,nonnegative_output,financial_floor,exercise_head,continuation_head,shared_backbone,test_mae,test_rmse,reported_financial_violation_rate,exercise_f1
0,Constrained floor residual,False,True,True,True,False,False,False,0.000102,0.000213,NaN,NaN
1,Price-only constrained residual,False,True,True,True,False,False,False,0.000102,0.000213,NaN,NaN
2,Non-negative premium,False,True,True,False,False,False,False,0.000205,0.000356,NaN,NaN
3,Unconstrained premium,False,True,False,False,False,False,False,0.000220,0.000359,NaN,NaN
4,Multi-task constrained residual,False,True,True,True,True,False,True,0.000242,0.000504,NaN,NaN
5,Final integrated constrained price,False,True,True,True,True,True,True,0.000562,0.001122,NaN,NaN
6,Direct MLP,True,False,True,False,False,False,False,0.000782,0.001146,NaN,NaN
7,Final integrated direct head,True,False,True,False,True,True,True,0.005621,0.007370,NaN,NaN
8,Black–Scholes proxy,False,False,False,False,False,False,False,0.013398,0.027384,NaN,NaN


## 10. Classical and neural Longstaff–Schwartz

These results remain separate from the static table. MAE and RMSE are measured against CRR contract prices, while confidence-interval coverage, policy behavior, training cost, and valuation runtime provide additional evidence.


In [10]:
lsm_comparison = build_lsm_comparison(
    project_results
)

if STRICT_MODE and lsm_comparison.empty:
    raise RuntimeError(
        "LSM comparison is empty."
    )

display(lsm_comparison)


,mae,maximum_absolute_error,mean_bias,median_absolute_error,method,normalized_mae,rmse,ci_coverage,valuation_seconds,training_seconds
0,0.039500,0.126719,-0.034377,0.031453,classical_lsm_price,0.004007,0.051623,0.88,0.082729,NaN
1,0.087199,0.400425,-0.084350,0.063802,neural_lsm_price,0.006911,0.121772,0.66,0.075390,628.049543


## 11. Automatically derived H1–H6 evidence and decisions

The evidence dictionary is generated from the consolidated tables and canonical packages. No manually edited intermediary JSON file is required.


In [11]:
hypothesis_evidence = build_hypothesis_evidence(
    project_results,
    static_pricing,
    financial_consistency,
    ood_results,
    runtime_comparison,
)

write_json(
    FINAL_OUTPUT_DIR / "hypothesis_evidence.json",
    hypothesis_evidence,
)

hypothesis_decisions = decide_all_hypotheses(
    hypothesis_evidence
)

display(
    pd.Series(
        hypothesis_evidence,
        name="value",
    ).to_frame()
)
display(hypothesis_decisions)

if (
    STRICT_MODE
    and hypothesis_decisions[
        "decision"
    ].eq("Inconclusive").any()
):
    unresolved = hypothesis_decisions.loc[
        hypothesis_decisions[
            "decision"
        ].eq("Inconclusive"),
        "hypothesis",
    ].tolist()
    raise RuntimeError(
        "Strict final evaluation has unresolved "
        f"hypotheses: {unresolved}"
    )


,value
black_scholes_mae,0.013398
direct_mlp_mae,0.000782
best_residual_mae,0.000102
direct_violation_rate,NaN
constrained_violation_rate,NaN
price_only_boundary_f1,0.996393
multitask_boundary_f1,0.996388
price_only_boundary_error,0.000079
multitask_boundary_error,0.000151
required_h4_f1_gain,0.01


,hypothesis,decision,primary_evidence,secondary_evidence,limitation
0,H1,Supported,Direct MLP MAE / Black–Scholes proxy MAE = 0.0...,Supported requires a ratio no greater than 0.95.,The decision concerns the fixed in-domain test...
1,H2,Supported,Best residual-model MAE / direct-model MAE = 0...,Supported requires a ratio no greater than 0.98.,The residual configuration must have been sele...
2,H3,Inconclusive,Comparable financial-violation rates are unava...,,No result is inferred from missing evidence.
3,H4,Not supported,Boundary F1 gain=-0.000006; relative boundary-...,Required F1 gain=0.010000; required relative e...,Boundary quality depends on the CRR label reso...
4,H5,Partially supported,Neural LSM marginal runtime / CRR runtime = 0....,Supported requires a ratio no greater than 0.50.,Data generation and policy-training cost are r...
5,H6,Supported,Final integrated constrained price: aggregate ...,Supported requires a deterioration ratio of at...,An aggregate result can conceal regime-specifi...


RuntimeError: Strict final evaluation has unresolved hypotheses: ['H3']

## 12. Literature-synthesis handoff

The code does not invent citation links. It exports each empirical finding with an explicit manual citation-mapping status so the final written report can connect the findings to the supplied papers.


In [ ]:
literature_synthesis = (
    build_literature_handoff(
        hypothesis_decisions
    )
)

display(literature_synthesis)


## 13. Export final evaluation

Strict export rejects missing required artifacts, empty tables, all-missing tables, and inconclusive hypotheses. A successful run therefore produces a defensible final evidence package rather than a skeleton.


In [ ]:
tables_to_export = {
    "consolidated_model_metrics": static_pricing,
    "metric_inventory": metric_inventory,
    "financial_consistency_table": (
        financial_consistency
    ),
    "boundary_comparison": boundary_comparison,
    "ood_results": ood_results,
    "ood_model_summary": ood_summary,
    "runtime_comparison": runtime_comparison,
    "static_model_ablation": static_ablation,
    "lsm_comparison": lsm_comparison,
    "literature_synthesis": literature_synthesis,
}

status = "READY_FOR_FINAL_WRITEUP"

exported_paths = export_final_evaluation(
    FINAL_OUTPUT_DIR,
    tables=tables_to_export,
    hypothesis_decisions=hypothesis_decisions,
    artifact_audit=artifact_audit,
    summary={
        "status": status,
        "valid_required_artifacts": (
            valid_required
        ),
        "required_artifacts": total_required,
        "expected_total_observations": (
            design.get(
                "expected_total_observations"
            )
        ),
        "best_static_model": (
            best_static["model"]
        ),
        "best_static_mae": (
            best_static["mae"]
        ),
    },
    strict=STRICT_MODE,
)

display(
    pd.Series(
        exported_paths,
        name="path",
    ).to_frame()
)


# Final project conclusion

A successful strict execution establishes that:

- every required upstream artifact is present and schema-compatible;
- static-model results are compared on one common normalized-price test split;
- financial consistency, exercise behavior, OOD deterioration, and runtime are populated from explicit source packages;
- classical and neural Longstaff–Schwartz results are reported on their own contract-level basis;
- H1–H6 decisions are derived automatically from the exported evidence;
- the final write-up package contains no empty placeholder tables.

The remaining work is interpretive writing and manual mapping of the supplied research papers to the empirical findings.
